# Aula 24 — Capstone P5: MLP manual em MNIST

Laboratório da [aula](../aulas/24-capstone-p5.md). Implementação exclusivamente em NumPy: nenhuma biblioteca de redes neurais ou diferenciação automática.

## Objetivo e protocolo fixado antes dos resultados

Treinar uma MLP 784 → 64 → 10, auditar suas derivadas, comparar um baseline linear e três ablações. Usamos 10.000 imagens de treino e 2.000 de validação, retiradas de forma estratificada das 60.000 imagens oficiais de treino. As 48.000 restantes ficam fora deste protocolo econômico. As 10.000 imagens oficiais de teste só entram no cálculo de métricas depois da seleção.

As configurações usam 15 épocas, lotes de até 128, learning rate 0,05, momentum 0,9 e seeds 101, 202 e 303. A escolha é pela média da menor cross-entropy de validação de cada seed. A seed do modelo final é **101, definida antecipadamente**. Não há promessa de ranking entre ablações ou acurácia mínima no teste.

## Ambiente e execução

Python >= 3.10, NumPy >= 1.24 e Matplotlib >= 3.7. Para validar o formato: nbformat >= 5.7. Testado com Python 3.12.14, NumPy 2.3.5 e Matplotlib 3.10.8. Instalação, se necessária: `python -m pip install 'numpy>=1.24' 'matplotlib>=3.7' 'nbformat>=5.7'`.

Execute todas as células em ordem em um kernel novo. CPU é suficiente; o tempo depende da BLAS. Os quatro arquivos gzip somam cerca de 12 MB. Cache, relatório e figuras são gravados na pasta `p5_mnist_saida/` do diretório de execução, sem sobrescrever os assets publicados. Não há credenciais. Os outputs são removidos do notebook versionado após a execução de validação.

### Resultado da execução de referência

A regra de validação selecionou `init_pequena` por uma diferença de apenas 0,000024 na CE média frente à referência, muito abaixo da variação entre três seeds. A seed final 101, fixada previamente, alcançou CE 0,143557 e acurácia 95,70% no teste oficial; o baseline linear atingiu 91,21%. O pior erro relativo por tensor foi 7,253 × 10⁻¹¹. As 10 auditorias passaram. Esses números descrevem o protocolo reduzido; não demonstram superioridade universal da inicialização pequena.


In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
import sys, copy, json, gzip, struct, hashlib, warnings
from pathlib import Path
from urllib.request import urlopen
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

warnings.simplefilter("error")
np.seterr(divide="raise", over="raise", invalid="raise")
OUT = Path("p5_mnist_saida")
OUT.mkdir(exist_ok=True)
SEEDS = (101, 202, 303)
SPLIT_SEED = 20260924
EPOCHS, BATCH, LR, MOMENTUM = 15, 128, 0.05, 0.9
audit = {}
print({"python": sys.version.split()[0], "numpy": np.__version__, "matplotlib": matplotlib.__version__})

## 1. Fonte e integridade do MNIST

Os checksums MD5 são os registrados no [carregador oficial do torchvision](https://github.com/pytorch/vision/blob/main/torchvision/datasets/mnist.py), consultado em 09/09/2026. **Torchvision não é importado**: serve apenas como referência de identidade dos arquivos. O espelho HTTPS do Google fornece os mesmos bytes. MD5 aqui detecta corrupção/versão, não comprova autenticidade criptográfica; HTTPS e procedência permanecem necessários. Registramos também SHA-256 dos arquivos recebidos.

O parser verifica o formato IDX big-endian, dimensões, contagens e intervalo dos rótulos. A leitura dos arquivos de teste é adiada para a seção final; baixar seus bytes não calcula nenhuma métrica.

In [ ]:
RESOURCES = {
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}
BASE_URL = "https://storage.googleapis.com/cvdf-datasets/mnist/"
def obtain(item):
    name, expected = item
    path = OUT / name
    if not path.exists():
        with urlopen(BASE_URL + name, timeout=60) as response:
            payload = response.read()
        assert hashlib.md5(payload).hexdigest() == expected, name
        path.write_bytes(payload)
    payload = path.read_bytes()
    assert hashlib.md5(payload).hexdigest() == expected, name
    return name, hashlib.sha256(payload).hexdigest()

with ThreadPoolExecutor(max_workers=4) as pool:
    source_hashes = dict(pool.map(obtain, RESOURCES.items()))

def read_idx(name, images, expected_n):
    payload = gzip.decompress((OUT / name).read_bytes())
    if images:
        magic, count, rows, cols = struct.unpack(">IIII", payload[:16])
        assert (magic, count, rows, cols) == (2051, expected_n, 28, 28)
        assert len(payload) == 16 + count * rows * cols
        return np.frombuffer(payload, dtype=np.uint8, offset=16).reshape(count, 784)
    magic, count = struct.unpack(">II", payload[:8])
    assert (magic, count) == (2049, expected_n)
    assert len(payload) == 8 + count
    labels = np.frombuffer(payload, dtype=np.uint8, offset=8).astype(np.int64)
    assert np.all((labels >= 0) & (labels < 10))
    return labels

raw_train = read_idx("train-images-idx3-ubyte.gz", True, 60000)
labels_train = read_idx("train-labels-idx1-ubyte.gz", False, 60000)
audit["fontes_idx"] = len(source_hashes) == 4
print("IDX de treino:", raw_train.shape, labels_train.shape)

## 2. Split e transformação sem vazamento

Cada classe fornece 1.000 imagens de treino e 200 de validação, sem reposição. O identificador é a posição original no IDX de treino, em um namespace distinto do teste. Guardamos os índices e seus hashes para auditoria. Não há identificador de escritor no IDX; portanto não afirmamos ter criado um split por pessoa. Duplicatas visuais entre arquivos também não são investigadas neste laboratório.

Dividir por 255 é uma transformação fixa. A média por pixel é ajustada somente nas 10.000 imagens de treino. Pixels constantes continuam válidos: não dividimos por seus desvios. A mesma média será reaplicada ao teste. O balanceamento deliberado de treino/validação e a distribuição natural do teste fazem parte do protocolo.

In [ ]:
split_rng = np.random.default_rng(SPLIT_SEED)
train_ids, val_ids = [], []
for label in range(10):
    ids = split_rng.permutation(np.flatnonzero(labels_train == label))
    train_ids.extend(ids[:1000])
    val_ids.extend(ids[1000:1200])
train_ids = split_rng.permutation(train_ids).astype(np.int64)
val_ids = split_rng.permutation(val_ids).astype(np.int64)
assert len(set(train_ids) & set(val_ids)) == 0
assert len(np.unique(train_ids)) == 10000 and len(np.unique(val_ids)) == 2000
pixel_mean = (raw_train[train_ids].astype(np.float64) / 255).mean(axis=0)
def transform(raw):
    return raw.astype(np.float64) / 255 - pixel_mean
Xtr, Xva = transform(raw_train[train_ids]), transform(raw_train[val_ids])
ytr, yva = labels_train[train_ids], labels_train[val_ids]
assert np.all(np.bincount(ytr) == 1000) and np.all(np.bincount(yva) == 200)
assert np.max(np.abs(Xtr.mean(axis=0))) < 1e-12
def hash_indices(ids):
    return hashlib.sha256(ids.astype("<i8").tobytes()).hexdigest()
split_manifest = {"seed": SPLIT_SEED, "train_ids": train_ids.tolist(), "val_ids": val_ids.tolist(),
                  "train_sha256": hash_indices(train_ids), "val_sha256": hash_indices(val_ids)}
(OUT / "split.json").write_text(json.dumps(split_manifest), encoding="utf-8")
audit["split_sem_intersecao"] = True
print("Treino/validação:", Xtr.shape, Xva.shape)

## 3. Forward, loss e backward manuais

Uma linha representa uma imagem. `W1` tem shape `(784,64)` e `W2`, `(64,10)`; os biases são vetores. A função também aceita uma rede menor para auditoria e um modelo linear sem camada oculta.

Usamos cross-entropy média por exemplo e L2 apenas nos pesos: `CE + lambda/2 * soma(W**2)`. O gradiente da CE é dividido pelo tamanho real do lote exatamente uma vez. `logp` vem de log-sum-exp estabilizado; a probabilidade não passa por clipping que altere a derivada. A loss retorna CE e penalidade separadamente. Todos os cálculos são float64.

In [ ]:
def initialize(d, h, c, seed, init="he"):
    rng = np.random.default_rng(seed)
    if h == 0:
        return {"W": rng.normal(0, np.sqrt(2 / (d + c)), (d, c)), "b": np.zeros(c)}
    scale = np.sqrt(2 / d) if init == "he" else 0.01
    return {"W1": rng.normal(0, scale, (d, h)), "b1": np.zeros(h),
            "W2": rng.normal(0, np.sqrt(2 / (h + c)), (h, c)), "b2": np.zeros(c)}

def objective(params, X, y, activation="relu", l2=0.0, backward=True):
    assert X.ndim == 2 and y.shape == (len(X),) and len(X) > 0
    if "W1" in params:
        z = X @ params["W1"] + params["b1"]
        a = np.maximum(z, 0) if activation == "relu" else np.tanh(z)
        logits = a @ params["W2"] + params["b2"]
    else:
        logits = X @ params["W"] + params["b"]
    shifted = logits - logits.max(axis=1, keepdims=True)
    logp = shifted - np.log(np.exp(shifted).sum(axis=1, keepdims=True))
    ce = -logp[np.arange(len(y)), y].mean()
    penalty = 0.5 * l2 * sum(np.sum(v * v) for k, v in params.items() if k.startswith("W"))
    if not backward:
        return float(ce), float(penalty), logp
    delta = np.exp(logp)
    delta[np.arange(len(y)), y] -= 1
    delta /= len(y)
    if "W1" in params:
        da = delta @ params["W2"].T
        dz = da * ((z > 0) if activation == "relu" else (1 - a * a))
        grads = {"W1": X.T @ dz + l2 * params["W1"], "b1": dz.sum(axis=0),
                 "W2": a.T @ delta + l2 * params["W2"], "b2": delta.sum(axis=0)}
    else:
        grads = {"W": X.T @ delta + l2 * params["W"], "b": delta.sum(axis=0)}
    assert all(grads[k].shape == v.shape for k, v in params.items())
    assert np.isfinite(ce + penalty) and all(np.isfinite(g).all() for g in grads.values())
    return float(ce), float(penalty), grads

def evaluate(params, X, y, activation="relu"):
    ce, _, logp = objective(params, X, y, activation, backward=False)
    return ce, float(np.mean(logp.argmax(axis=1) == y))

def update(params, velocity, grads, lr, momentum):
    assert params.keys() == velocity.keys() == grads.keys()
    for k in params:
        velocity[k] = momentum * velocity[k] - lr * grads[k]
        params[k] += velocity[k]

## 4. Gradient checking de cada tensor e das duas ativações

Verificamos **todas as 43 coordenadas** de uma rede 4 → 5 → 3, incluindo L2, com diferenças centrais. O erro relativo por tensor é `norma(g-num)/(norma(g)+norma(num)+1e-12)`; também declaramos o erro absoluto máximo. Para ReLU, confirmamos que as perturbações não cruzam zero. A verificação da rede reduzida valida as fórmulas; não é uma varredura exaustiva dos 50.890 parâmetros da rede MNIST.

In [ ]:
tiny_rng = np.random.default_rng(2401)
Xcheck = tiny_rng.normal(size=(6, 4))
ycheck = np.array([0, 1, 2, 0, 2, 1])
pcheck = initialize(4, 5, 3, 2402)
assert sum(v.size for v in pcheck.values()) == 43
gradient_report = {}
for activation in ("relu", "tanh"):
    _, _, analytic = objective(pcheck, Xcheck, ycheck, activation, 0.003)
    gradient_report[activation] = {}
    for name, value in pcheck.items():
        numerical = np.zeros_like(value)
        for index in np.ndindex(value.shape):
            old = value[index]
            h = 1e-5 * max(1.0, abs(old))
            signs = []
            losses = []
            for shift in (h, -h):
                value[index] = old + shift
                ce, reg, _ = objective(pcheck, Xcheck, ycheck, activation, 0.003, False)
                losses.append(ce + reg)
                signs.append(Xcheck @ pcheck["W1"] + pcheck["b1"] > 0)
            value[index] = old
            if activation == "relu":
                assert np.array_equal(signs[0], signs[1]), "Perturbação cruza quina"
            numerical[index] = (losses[0] - losses[1]) / (2 * h)
        relative = np.linalg.norm(analytic[name] - numerical) / (np.linalg.norm(analytic[name]) + np.linalg.norm(numerical) + 1e-12)
        absolute = np.max(np.abs(analytic[name] - numerical))
        assert relative < 1e-7 and absolute < 1e-7
        gradient_report[activation][name] = {"relative": float(relative), "max_absolute": float(absolute)}
audit["gradientes_relu_tanh_l2"] = True
print(json.dumps(gradient_report, indent=2))

## 5. Contraprovas: média, atualização e sanidade

Duplicar um lote preserva CE média e gradientes. SGD é o caso `momentum=0`. Um gradiente com sinal invertido deve aumentar uma loss quadrática em um passo pequeno. Esses testes não substituem a checagem numérica; cobrem classes diferentes de defeitos.

In [ ]:
_, _, ga = objective(pcheck, Xcheck, ycheck, "tanh", 0.003)
_, _, gb = objective(pcheck, np.repeat(Xcheck, 2, axis=0), np.repeat(ycheck, 2), "tanh", 0.003)
assert all(np.allclose(ga[k], gb[k], atol=1e-14) for k in ga)
q = {"W": np.array([2.0, -1.0])}
v = {"W": np.zeros(2)}
before = q["W"].copy()
update(q, v, {"W": before.copy()}, 0.1, 0.0)
assert np.allclose(q["W"], 0.9 * before)
assert np.sum((before + 0.1 * before)**2) > np.sum(before**2)
audit["media_sgd_sinal"] = True
print("Média e SGD: contratos aprovados")

## 6. Laço de treino e checkpoint por época

Uma época usa todos os índices, incluindo o último lote de 16 imagens. Atualizamos o momentum com o gradiente do lote real. Medimos CE de treino e validação ao final da época, no **mesmo estado dos pesos**. A norma de gradiente registrada é a média ponderada das normas dos gradientes de cada lote (inclui L2); não é a norma de um único gradiente agregado.

O checkpoint inclui pesos, velocidades, estado do RNG, número da época, melhor estado e histórico. A função continua no limite entre épocas; retomada no meio de um lote exigiria também permutação e cursor. Não usamos dropout/normalização interna no modelo de referência para manter a integração inspecionável.

In [ ]:
def new_state(config, seed, d=784, classes=10):
    params = initialize(d, config["hidden"], classes, seed, config.get("init", "he"))
    return {"params": params, "velocity": {k: np.zeros_like(v) for k, v in params.items()},
            "rng": np.random.default_rng(seed + 10000).bit_generator.state,
            "epoch": 0, "best_ce": float("inf"), "best_epoch": 0, "best": None, "history": []}

def train(config, seed, X, y, V, vy, epochs, state=None):
    state = new_state(config, seed, X.shape[1], int(max(y.max(), vy.max())) + 1) if state is None else copy.deepcopy(state)
    rng = np.random.default_rng()
    rng.bit_generator.state = copy.deepcopy(state["rng"])
    activation, l2 = config["activation"], config["l2"]
    for _ in range(epochs):
        order = rng.permutation(len(X))
        norms = {k: 0.0 for k in state["params"] if k.startswith("W")}
        seen = 0
        for start in range(0, len(X), BATCH):
            ids = order[start:start + BATCH]
            _, _, grads = objective(state["params"], X[ids], y[ids], activation, l2)
            for k in norms:
                norms[k] += len(ids) * float(np.linalg.norm(grads[k]))
            update(state["params"], state["velocity"], grads, LR, MOMENTUM)
            seen += len(ids)
        assert seen == len(X)
        state["epoch"] += 1
        trce, tracc = evaluate(state["params"], X, y, activation)
        vce, vacc = evaluate(state["params"], V, vy, activation)
        row = {"epoch": state["epoch"], "train_ce": trce, "val_ce": vce,
               "train_accuracy": tracc, "val_accuracy": vacc,
               "gradient_norm": {k: value / seen for k, value in norms.items()}}
        state["history"].append(row)
        if vce < state["best_ce"]:
            state["best_ce"], state["best_epoch"] = vce, state["epoch"]
            state["best"] = copy.deepcopy(state["params"])
    state["rng"] = copy.deepcopy(rng.bit_generator.state)
    return state

reference = {"hidden": 64, "activation": "relu", "init": "he", "l2": 1e-4}
short = train(reference, 101, Xtr[:256], ytr[:256], Xva[:100], yva[:100], 2)
resumed = train(reference, 101, Xtr[:256], ytr[:256], Xva[:100], yva[:100], 2, short)
continuous = train(reference, 101, Xtr[:256], ytr[:256], Xva[:100], yva[:100], 4)
assert all(np.array_equal(resumed["params"][k], continuous["params"][k]) for k in resumed["params"])
assert resumed["history"] == continuous["history"]
audit["retomada_exata"] = True
print("Retomada por época: idêntica ao treino contínuo")

## 7. Memorizar um lote pequeno antes do experimento

Usamos duas imagens por classe (20 no total), sem L2. É uma auditoria de capacidade de ajuste e do laço de treino, não uma medida de generalização. O orçamento é fixado em 600 passos antes da execução.

In [ ]:
small_ids = np.concatenate([np.flatnonzero(ytr == k)[:2] for k in range(10)])
psmall = initialize(784, 64, 10, 24)
vsmall = {k: np.zeros_like(v) for k, v in psmall.items()}
for step in range(600):
    _, _, grads = objective(psmall, Xtr[small_ids], ytr[small_ids])
    update(psmall, vsmall, grads, 0.05, 0.9)
small_ce, small_acc = evaluate(psmall, Xtr[small_ids], ytr[small_ids])
assert small_ce < 0.02 and small_acc == 1.0
audit["overfit_20"] = True
print(f"20 imagens: CE={small_ce:.8f}; acurácia={small_acc:.3f}")

## 8. Ablações e baseline sem consultar o teste

As quatro MLPs alteram um fator em relação à referência. `init_pequena` muda apenas a escala de W1 para 0,01; W2 mantém Xavier. `tanh` preserva exatamente os pesos iniciais da referência: é uma ablação de ativação sob inicialização controlada, **não** uma busca otimizada para tanh. `sem_l2` zera a penalidade. As três seeds usam os mesmos splits e a mesma ordem dos lotes entre configurações.

O baseline linear softmax usa o mesmo otimizador, L2, épocas e critério de checkpoint; tem menos parâmetros. Igualar épocas/atualizações não iguala FLOPs. Nenhuma configuração é adicionada depois de olhar resultados. A validação seleciona tanto a época quanto a configuração; esse reuso gera viés de seleção, a ser avaliado no teste reservado.

In [ ]:
configs = {
    "referencia": reference,
    "init_pequena": {**reference, "init": "small"},
    "tanh": {**reference, "activation": "tanh"},
    "sem_l2": {**reference, "l2": 0.0},
}
experiments = {}
for name, config in configs.items():
    experiments[name] = {}
    for seed in SEEDS:
        experiments[name][seed] = train(config, seed, Xtr, ytr, Xva, yva, EPOCHS)
    values = [experiments[name][seed]["best_ce"] for seed in SEEDS]
    print(name, "CE val média/desvio:", f"{np.mean(values):.6f}", f"{np.std(values, ddof=1):.6f}", flush=True)

baseline_config = {"hidden": 0, "activation": "relu", "l2": 1e-4}
baseline = train(baseline_config, SEEDS[0], Xtr, ytr, Xva, yva, EPOCHS)
selection = {name: float(np.mean([runs[s]["best_ce"] for s in SEEDS])) for name, runs in experiments.items()}
chosen = min(selection, key=selection.get)
chosen_config = configs[chosen]
chosen_state = experiments[chosen][SEEDS[0]]
final_params = copy.deepcopy(chosen_state["best"])
assert sum(v.size for v in final_params.values()) == 50890
audit["protocolo_12_runs"] = sum(len(v) for v in experiments.values()) == 12
print("Escolha congelada:", chosen, "seed", SEEDS[0], "época", chosen_state["best_epoch"])
print("Baseline: CE val", round(baseline["best_ce"], 6), "época", baseline["best_epoch"])

## 9. Curvas e fluxo de gradientes

As curvas mostram a execução da seed 101 da configuração escolhida, incluindo todas as 15 épocas executadas. A linha vertical identifica o estado que será usado no teste. As normas incluem todos os lotes de cada época; valores não nulos não provam que toda unidade está aprendendo. Para localizar um defeito, inspecione também distribuições de ativações e gradientes, como nas aulas anteriores.

In [ ]:
history = chosen_state["history"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
epochs_axis = [row["epoch"] for row in history]
axes[0].plot(epochs_axis, [r["train_ce"] for r in history], "o-", label="Treino")
axes[0].plot(epochs_axis, [r["val_ce"] for r in history], "s--", label="Validação")
axes[0].axvline(chosen_state["best_epoch"], color="black", linestyle=":", label="Checkpoint")
axes[0].set(xlabel="Época", ylabel="Cross-entropy média", title="Loss de dados em pesos fixos")
for key, style in (("W1", "o-"), ("W2", "s--")):
    axes[1].semilogy(epochs_axis, [r["gradient_norm"][key] for r in history], style, label=key)
axes[1].set(xlabel="Época", ylabel="Média ponderada da norma L2", title="Gradientes por camada")
for ax in axes:
    ax.legend(); ax.grid(alpha=0.25)
fig.savefig(OUT / "24-capstone-p5-curvas.png", dpi=140)
plt.show()
plt.close(fig)
assert all(np.isfinite(list(r["gradient_norm"].values())).all() for r in history)
audit["curvas_gradientes_finitos"] = True

## 10. Avaliação única no teste oficial

Só agora carregamos imagens e rótulos de teste para inferência. Avaliamos a MLP escolhida e o baseline definido antecipadamente. A matriz de confusão tem linhas = classe real e colunas = predita. Macro-F1 dá o mesmo peso a cada classe; acurácia conta imagens. A restauração usa os melhores pesos de validação, sem novo treinamento no teste.

Não selecione outra configuração após esta célula. Uma mudança de protocolo exige uma nova avaliação independente; repetir escolhas contra este mesmo teste passa a usá-lo como validação.

In [ ]:
raw_test = read_idx("t10k-images-idx3-ubyte.gz", True, 10000)
ytest = read_idx("t10k-labels-idx1-ubyte.gz", False, 10000)
Xtest = transform(raw_test)

def test_metrics(params, activation):
    ce, _, logp = objective(params, Xtest, ytest, activation, backward=False)
    prediction = logp.argmax(axis=1)
    confusion = np.zeros((10, 10), dtype=np.int64)
    np.add.at(confusion, (ytest, prediction), 1)
    tp = confusion.diagonal()
    denom = confusion.sum(axis=0) + confusion.sum(axis=1)
    f1 = np.divide(2 * tp, denom, out=np.zeros(10, dtype=float), where=denom > 0)
    assert confusion.sum() == 10000
    assert np.isclose(np.trace(confusion) / confusion.sum(), np.mean(prediction == ytest))
    return {"ce": ce, "accuracy": float(np.mean(prediction == ytest)), "macro_f1": float(f1.mean()),
            "per_class_f1": f1.tolist(), "confusion": confusion.tolist()}, prediction, logp

test_result, prediction, test_logp = test_metrics(final_params, chosen_config["activation"])
baseline_result, _, _ = test_metrics(baseline["best"], "relu")
audit["metricas_teste_coerentes"] = True
print("MLP:", {k: v for k, v in test_result.items() if k in ("ce", "accuracy", "macro_f1")})
print("Linear:", {k: v for k, v in baseline_result.items() if k in ("ce", "accuracy", "macro_f1")})

## 11. Erros inspecionáveis e limites

Mostramos as primeiras oito imagens incorretas na ordem oficial, sem escolher casos visualmente convenientes. Cada título traz índice, rótulo real, predição e probabilidade softmax. Probabilidade elevada não é garantia de acerto nem medida de calibração. A imagem ajuda a formular hipóteses sobre forma, ambiguidade e pré-processamento; não autoriza corrigir rótulos sem auditoria.

O experimento não demonstra robustez fora de MNIST, equivalência com treinamento nas 60.000 imagens, superioridade universal da ativação vencedora, privacidade, causalidade das diferenças ou implantação pronta. Uma MLP ignora explicitamente a geometria espacial que CNNs explorarão mais adiante.

In [ ]:
wrong = np.flatnonzero(prediction != ytest)
examples = wrong[:8]
fig, axes = plt.subplots(2, 4, figsize=(10, 5), layout="constrained")
for ax, index in zip(axes.flat, examples):
    probability = float(np.exp(test_logp[index, prediction[index]]))
    ax.imshow(raw_test[index].reshape(28, 28), cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"#{index} real={ytest[index]} → {prediction[index]}\np={probability:.2f}")
    ax.axis("off")
for ax in list(axes.flat)[len(examples):]:
    ax.axis("off")
fig.savefig(OUT / "24-capstone-p5-erros.png", dpi=140)
plt.show()
plt.close(fig)
confusion = np.array(test_result["confusion"])
off_diagonal = confusion.copy()
np.fill_diagonal(off_diagonal, 0)
largest = np.unravel_index(off_diagonal.argmax(), off_diagonal.shape)
print("Erros:", len(wrong), "Maior confusão dirigida:", largest, int(off_diagonal[largest]))

## 12. Handoff e relatório auditável

Exportamos pesos de inferência, média por pixel e uma fixture de forward/backward em `.npz`, sem pickle. O teste de ida e volta exige logits idênticos. Este pacote de inferência **não** é checkpoint para continuar treinamento: o estado completo do otimizador foi exercitado na seção 6. Para transportar um checkpoint de treino, serialize também velocidade, RNG e contador junto com a configuração.

O relatório inclui protocolo, identidade dos dados, hashes dos splits, derivadas, resultados individuais de validação, curvas e métricas de teste. As fixtures usam imagens de treino, não de teste. No M6 elas permitirão comparar as operações manuais com autograd; não implementamos PyTorch neste capstone.

In [ ]:
fixture_X, fixture_y = Xtr[:3].copy(), ytr[:3].copy()
_, _, fixture_grads = objective(final_params, fixture_X, fixture_y, chosen_config["activation"], chosen_config["l2"])
_, _, fixture_logp = objective(final_params, fixture_X, fixture_y, chosen_config["activation"], backward=False)
np.savez(OUT / "p5_inferencia_fixture.npz", **final_params, pixel_mean=pixel_mean,
         fixture_X=fixture_X, fixture_y=fixture_y, fixture_logp=fixture_logp,
         **{"grad_" + k: v for k, v in fixture_grads.items()})
with np.load(OUT / "p5_inferencia_fixture.npz", allow_pickle=False) as archive:
    restored = {k: archive[k].copy() for k in final_params}
    assert np.array_equal(archive["pixel_mean"], pixel_mean)
_, _, roundtrip_logp = objective(restored, fixture_X, fixture_y, chosen_config["activation"], backward=False)
assert np.array_equal(roundtrip_logp, fixture_logp)
audit["exportacao_sem_pickle"] = True

report = {
    "dataset": "MNIST", "protocol": {"train": 10000, "validation": 2000, "test": 10000,
    "unused_official_train": 48000, "split_seed": SPLIT_SEED, "seeds": list(SEEDS),
    "epochs": EPOCHS, "batch_size": BATCH, "learning_rate": LR, "momentum": MOMENTUM,
    "dtype": "float64", "selection": "mean best validation CE; final seed fixed to 101"},
    "environment": {"python": sys.version.split()[0], "numpy": np.__version__, "matplotlib": matplotlib.__version__},
    "source_base_url": BASE_URL, "source_sha256": source_hashes,
    "split_hashes": {"train": hash_indices(train_ids), "validation": hash_indices(val_ids)},
    "gradient_checks": gradient_report, "tiny_overfit": {"ce": small_ce, "accuracy": small_acc},
    "configs": configs, "validation": {name: [{"seed": seed, "ce": runs[seed]["best_ce"],
        "best_epoch": runs[seed]["best_epoch"]} for seed in SEEDS] for name, runs in experiments.items()},
    "selected": chosen, "final_seed": SEEDS[0], "best_epoch": chosen_state["best_epoch"],
    "selected_history": history, "baseline_history": baseline["history"],
    "baseline_validation_ce": baseline["best_ce"], "test": test_result, "baseline_test": baseline_result,
    "first_errors": [{"index": int(i), "true": int(ytest[i]), "predicted": int(prediction[i]),
        "probability": float(np.exp(test_logp[i, prediction[i]]))} for i in examples],
    "largest_confusion": {"true": int(largest[0]), "predicted": int(largest[1]), "count": int(off_diagonal[largest])},
    "audits": audit,
}
assert all(audit.values())
(OUT / "24-capstone-p5-resultados.json").write_text(json.dumps(report, indent=2, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8")
print("Auditorias:", len(audit), "/", len(audit))
print("Relatório e artefatos:", OUT.resolve())

## Exercícios e próximos passos

1. Por que uma CE menor no treino não basta para selecionar o modelo? **Resposta:** ajuste ao treino não estima generalização; o protocolo seleciona pela validação e só depois mede o teste.
2. Quantos parâmetros possui a MLP? **Resposta:** `784*64 + 64 + 64*10 + 10 = 50.890`; o baseline possui `784*10 + 10 = 7.850`.
3. A ablação tanh prova que sua inicialização ideal é He? **Resposta:** não; manter W1 fixo isola a troca de ativação, mas não otimiza o par ativação/inicialização.
4. Como continuar o treino após carregar apenas o `.npz`? **Resposta:** não é uma retomada fiel. É necessário preservar a velocidade de momentum, RNG, época, configuração e histórico de seleção.
5. Uma maior probabilidade softmax nos erros prova calibração ruim? **Resposta:** casos individuais não bastam; exige avaliação específica de calibração e amostra representativa.

Próxima sequência: M6 — PyTorch. Reimplemente inicialmente o mesmo grafo e compare uma fixture fixa antes de trocar arquitetura, otimizador ou protocolo.

### Referências técnicas

- [Fonte oficial do loader MNIST e checksums](https://github.com/pytorch/vision/blob/main/torchvision/datasets/mnist.py), consulta em 09/09/2026; não é dependência de execução.
- [Stanford CS231n: checks e diagnóstico](https://cs231n.github.io/neural-networks-3/).
- [Goodfellow, Bengio e Courville: metodologia prática, cap. 11](https://www.deeplearningbook.org/contents/guidelines.html), livro de 2016.
- [Glorot e Bengio (2010)](https://proceedings.mlr.press/v9/glorot10a.html).

As figuras geradas têm descrição e interpretação na aula; os dados numéricos ficam no relatório JSON. O notebook versionado mantém outputs limpos por política do repositório.
- [He et al. (2015): inicialização para retificadores](https://arxiv.org/abs/1502.01852).
